In [ ]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd

In [ ]:
# https://github.com/Azure/mmlspark

In [ ]:
spark = (SparkSession.builder
    .appName("Churn Prediction with LightGBM on Spark")
    .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.13:1.1.3-spark4.1")
    .config("spark.jars.repositories", "https://mmlspark.blob.core.windows.net/maven")
    .config("spark.executor.cores", "2")
    .enableHiveSupport()
    .getOrCreate())

In [ ]:
spark.version

In [ ]:
# in namenode
%%bash
BASE=https://raw.githubusercontent.com/erkansirin78/datasets/master/churn-telecom/cell2celltrain_clean_parquet
DIR=/tmp/datasets/churn-telecom/cell2celltrain_clean_parquet
mkdir -p $DIR && cd $DIR
for f in part-00000-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet \
         part-00001-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet \
         part-00002-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet \
         part-00003-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet \
         _SUCCESS
do
  curl -sSL -o "$f" "$BASE/$f"
done
ls -l $DIR

# hdfs dfs -put  /tmp/datasets/ /tmp/

## Read train data

In [ ]:
df1 = spark.read.format("parquet") \
.load("/tmp/datasets/churn-telecom/cell2celltrain_clean_parquet/")

## Explore train data

In [ ]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [ ]:
# Since there are many columns we'd better explore part by part 
df1 \
.limit(46).toPandas()

In [ ]:
# Schema change analysis

# MonthlyRevenue, MonthlyMinutes, TotalRecurringCharge, OverageMinutes, PercChangeMinutes, PercChangeRevenues
# Handsets, HandsetModels, CurrentEquipmentDays, AgeHH1, AgeHH2  to integer

# DirectorAssistedCalls, HandsetPrice to float

In [ ]:
df1.printSchema()

In [ ]:
# Null check
for col_name, col_type in df1.dtypes:
    cond = F.col(col_name).isNull()
    if col_type == "string":
        cond = cond | (F.trim(F.col(col_name)) == "")
    null_count = df1.filter(cond).count()
    if null_count > 0:
        print("{} {} type has {} null values".format(col_name, col_type, null_count))

# Distribution of Output/Target variable

In [ ]:
df1.select("Churn").groupBy("Churn").count().show()

In [ ]:
! pip install matplotlib

In [ ]:
df1.select("Churn").groupBy("Churn").count().toPandas().plot(kind='bar')

# Data Preperation for the Model

In [ ]:
categoric_cols = []
numeric_cols = []
discarted_cols = []
label_col = []

In [ ]:
# column seperation
for col_name in df1.dtypes:
    if (col_name[0] not in []):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [ ]:
for col_name in categoric_cols:
    df1.groupBy(col_name).agg(F.count("*").alias("TotalCount")).orderBy(F.desc("TotalCount")).show()

In [ ]:
# column seperation after analysis
categoric_cols = []
numeric_cols = []
# Why we discard these? Because they have high cardinality. In other words they have too many distinct categories
# Adding them to model will not do good if not too bad.
discarted_cols = ['CustomerID', 'ServiceArea']
label_col = ['Churn']

In [ ]:
for col_name in df1.dtypes:
    if (col_name[0] not in discarted_cols+label_col):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [ ]:
print(categoric_cols)
print(len(categoric_cols))

In [ ]:
print(numeric_cols)
print(len(numeric_cols))

In [ ]:
print(discarted_cols+label_col)
print(len(discarted_cols+label_col))

## verification of columns

In [ ]:
if len(df1.columns) == (len(categoric_cols)+len(numeric_cols)+len(discarted_cols+label_col)):
    print("Columns verified")
else: print("Columns are not verified")

## Select columns for onehotencoder

In [ ]:
# Here we count the distinct categories in categoric columns. If there is more than two,
# we will add those to to_be_onehotencoded_cols list If there is just 2 we don't need to use onehotencoder
# So if distinct category gt 2 we have to add it to_be_onehotencoded_cols list
to_be_onehotencoded_cols = []

for col_name in categoric_cols:
    count = df1.select(col_name).distinct().count()
    if count > 2:
        to_be_onehotencoded_cols.append(col_name)
    print("{} has {} distinct category.".format(col_name, count))
    

In [ ]:
print(to_be_onehotencoded_cols)

# StringIndexer

In [ ]:
from pyspark.ml.feature import StringIndexer

In [ ]:
# Will hold stringIndexer objects and column names
my_dict = {}

# Will collect StringIndexer ojects
string_indexer_objs = []

# Will collect StringIndexer output colnames
string_indexer_output_names = []

# Will collect OneHotEncoder output colnames
ohe_col_input_names = []
ohe_col_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_indexobj"] = StringIndexer() \
    .setHandleInvalid('skip') \
    .setInputCol(col_name) \
    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_indexobj"))
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in to_be_onehotencoded_cols:
        ohe_col_input_names.append(col_name+"_indexed")
        ohe_col_output_names.append(col_name+"_ohe")

In [ ]:
print(string_indexer_objs)
print(len(string_indexer_objs))

In [ ]:
print(string_indexer_output_names)
print(len(string_indexer_output_names))

In [ ]:
print(ohe_col_input_names)
print(len(ohe_col_input_names))

In [ ]:
print(ohe_col_output_names)
print(len(ohe_col_output_names))

# OnehotEncoder

In [ ]:
from pyspark.ml.feature import OneHotEncoder

In [ ]:
encoder = OneHotEncoder() \
.setInputCols(ohe_col_input_names) \
.setOutputCols(ohe_col_output_names)

# VectorAssembler

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
# We have to exclude onehotencoded categoric cols from stringIndexer cols
string_indexer_col_names_ohe_exluded = list(set(string_indexer_output_names).difference(set(ohe_col_input_names)))
string_indexer_col_names_ohe_exluded

## column verification

In [ ]:
if len(categoric_cols) == (len(string_indexer_col_names_ohe_exluded)+len(ohe_col_output_names)):
    print("Columns verified")
else: print("Columns are not verified")

In [ ]:
assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+string_indexer_col_names_ohe_exluded+ohe_col_output_names) \
.setOutputCol('unscaled_features')

# LabelIndexer

In [ ]:
label_indexer = StringIndexer().setHandleInvalid("skip") \
.setInputCol(label_col[0]) \
.setOutputCol("label")

# Feature Scaling

In [ ]:
from pyspark.ml.feature import StandardScaler

In [ ]:
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("scaled_features")

# PCA

In [ ]:
from pyspark.ml.feature import PCA

In [ ]:
pca = PCA().setInputCol("scaled_features").setK(10).setOutputCol("features")

# Estimator

In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier

In [ ]:
estimator = LightGBMClassifier(learningRate=0.3,
                           numIterations=200,
                           numLeaves=20,
                           featuresCol='features',
                           labelCol='label',
                           isUnbalance=True)

# Pipeline

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
pipeline_obj = Pipeline().setStages(string_indexer_objs+[encoder, assembler, label_indexer, scaler, pca, estimator])

In [ ]:
# Split data

In [ ]:
train_df, test_df = df1.randomSplit([0.8, 0.2], seed=142)

In [ ]:
print(train_df.count())
print(test_df.count())

# Train Model

In [ ]:
pipeline_model = pipeline_obj.fit(train_df)

# Save the model

In [ ]:
pipeline_model.stages[-1].write().overwrite().save("saved_models/lightgbm_churn_pca10")

# Prediction

In [ ]:
transformed_df = pipeline_model.transform(test_df)

In [ ]:
transformed_df.select('label','prediction','probability','rawPrediction').show(truncate=False)

# Evaluate the model

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [ ]:
evaluator = BinaryClassificationEvaluator()

In [ ]:
evaluator.evaluate(transformed_df)

In [ ]:
evaluator.getMetricName()

In [ ]:
transformed_df.select('label','prediction','probability','rawPrediction') \
.filter("prediction == 1.0") \
.show(truncate=False)

In [ ]:
transformed_df.select('label','prediction','probability','rawPrediction') \
.withColumn("score", F.col('probability').cast('string')).show()

In [ ]:
transformed_df.select('label','prediction','probability','rawPrediction').printSchema()

# Get churn score from probability

In [ ]:
import pyspark.sql.functions as F
from pyspark.ml.functions import vector_to_array

# vector_to_array is a built-in JVM function: no Python UDF, so no Python worker
# on YARN nodes (avoids driver/worker Python version mismatch)
score_df = transformed_df.select('CustomerID','label','prediction','probability','rawPrediction') \
.withColumn('score', vector_to_array('probability')[1] * 1000)

In [ ]:
score_df.filter("prediction == 1.0").show()

# Top 20 Costumers most likely to churn

In [ ]:
score_df.orderBy(F.desc("score")).show()

In [ ]:
score_df.printSchema()

# Hyperparameter Tuning

In [ ]:
from pyspark.ml.tuning import CrossValidatorModel, ParamGridBuilder, CrossValidator

In [ ]:
# Spark 4 + SynapseML: do not tune pca.k here. CV models with different feature
# counts break LightGBM ("number of features in data (5) is not the same as it
# was in training data (10)"). Keep k fixed, tune LightGBM params only.
paramGrid = ParamGridBuilder() \
.addGrid(estimator.numLeaves, [5, 20]) \
.addGrid(estimator.learningRate, [0.1, 0.3]) \
.build()

In [ ]:
crossval = CrossValidator(estimator=pipeline_obj,
                          estimatorParamMaps=paramGrid,
                          evaluator=BinaryClassificationEvaluator(),
                          numFolds=5) 

In [ ]:
cvModel = crossval.fit(train_df)

In [ ]:
type(cvModel)

In [ ]:
cvModel.params

In [ ]:
bestModel = cvModel.bestModel

In [ ]:
evaluator.evaluate(bestModel.transform(df1))

In [ ]:
evaluator.evaluate(cvModel.transform(df1))

In [ ]:
spark.stop()